In [1]:
from abc import * 
from typing import TypedDict, List, Any
from langgraph.graph import StateGraph, START, END

In [2]:
# define langgraph messagetypes
class InputState(TypedDict):
    input_value: int

class OutputState(TypedDict):
    response: int
    history: List[str]

class OverallState(TypedDict):
    input_value: int
    response: int
    history: List[str]

In [3]:
# define Node Tempalte Class        
class NodeTemplate(metaclass=ABCMeta):
    def __init__(self):
        pass

    @abstractmethod
    def _pre_execute(self):
        pass 

    @abstractmethod
    def _post_execute(self):
        pass 
        
    def __call__(self, state:Any)->Any:
        print("-")
        print(state)
        print("-"*10)
        self.state = state
        self._pre_execute()
        self.result = self.execute()
        self._post_execute()
        return self.result
        
    @abstractmethod
    def execute(self)->OverallState:
        pass 

In [4]:
class Node1(NodeTemplate):
    def _pre_execute(self):
        print(f"{self.__class__.__name__} pre execute")
        print(self.state)
        
    def _post_execute(self):
        print(f"{self.__class__.__name__} post execute")
        print(self.state)
        
    def execute(self) -> OverallState:
        print("node1")
        self.state["input_value"] += 10
        if "history" in self.state:
            self.state["history"].append("node1")
        else:
            self.state["history"] = ["node1"]

        return {
            "input_value": self.state["input_value"],
            "response": self.state["input_value"],
            "history": self.state["history"],
        }       

In [5]:
class Node2(NodeTemplate):
    def _pre_execute(self):
        print(f"{self.__class__.__name__} pre execute")
        print(self.state)
        
    def _post_execute(self):
        print(f"{self.__class__.__name__} post execute")
        print(self.state)
        
    def execute(self) -> OverallState:
        print("node2")
        self.state["input_value"] += 10
        if "history" in self.state:
            self.state["history"].append("node2")
        else:
            self.state["history"] = ["node2"]

        return {
            "input_value": self.state["input_value"],
            "response": self.state["input_value"],
            "history": self.state["history"],
        }

In [6]:
class Node3(NodeTemplate):  
    def _pre_execute(self):
        print(f"{self.__class__.__name__} pre execute")
        print(self.state)
        
    def _post_execute(self):
        print(f"{self.__class__.__name__} post execute")
        print(self.state)
        
    def execute(self) -> OverallState:
        print("node3")
        self.state["input_value"] += 10
        if "history" in self.state:
            self.state["history"].append("node3")
        else:
            self.state["history"] = ["node3"]
        return {
            "input_value": self.state["input_value"],
            "response": self.state["input_value"],
            "history": self.state["history"],
        }

In [7]:
graph = StateGraph(OverallState, input=InputState, output=OverallState)
graph.add_node("node1", Node1())
graph.add_node("node2", Node2())
graph.add_node("node3", Node3())
graph.add_edge(START,'node1')
graph.add_edge('node1','node2')
graph.add_edge('node2','node3')
graph.add_edge('node3',END)
# graph.set_entry_point('node')
app = graph.compile()

In [8]:
print(app.get_graph().draw_mermaid())

%%{init: {'flowchart': {'curve': 'linear'}}}%%
graph TD;
	__start__([<p>__start__</p>]):::first
	node1(node1)
	node2(node2)
	node3(node3)
	__end__([<p>__end__</p>]):::last
	__start__ --> node1;
	node1 --> node2;
	node2 --> node3;
	node3 --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [9]:
res = app.invoke({"input_value":1})

-
{'input_value': 1}
----------
Node1 pre execute
{'input_value': 1}
node1
Node1 post execute
{'input_value': 11, 'history': ['node1']}
-
{'input_value': 11, 'response': 11, 'history': ['node1']}
----------
Node2 pre execute
{'input_value': 11, 'response': 11, 'history': ['node1']}
node2
Node2 post execute
{'input_value': 21, 'response': 11, 'history': ['node1', 'node2']}
-
{'input_value': 21, 'response': 21, 'history': ['node1', 'node2']}
----------
Node3 pre execute
{'input_value': 21, 'response': 21, 'history': ['node1', 'node2']}
node3
Node3 post execute
{'input_value': 31, 'response': 21, 'history': ['node1', 'node2', 'node3']}


In [10]:
print("---")
print(res)

---
{'input_value': 31, 'response': 31, 'history': ['node1', 'node2', 'node3']}
